<center>
    <img src="https://rockborne.com/wp-content/uploads/2021/07/LandingPage-Header-RED-CENTRE.jpg" width="900" alt="logo"  />
</center>

# Statistical Approaches to Data Quality

*Session 4 · Notebook 03 · Lecture · Student version*

## Overview

Real data is messy: it contains extreme values and gaps. This notebook covers the statistical tools for handling both, worked end to end on realistic datasets. We detect **outliers** with univariate methods (z-score, IQR) and a multivariate method (Mahalanobis distance) and then fix them, we classify why data is **missing** (the MCAR, MAR and MNAR mechanisms) and learn a concrete test for telling them apart, and we **impute** missing values with simple, KNN and MICE methods, weighing the trade-offs.

The concepts are general purpose; a dedicated section shows how they map onto risk analysis, and the exercises put them to work on real data.

## Learning Objectives

By the end of this notebook you will be able to:

- Detect outliers with the z-score and IQR methods, and handle them by removing or capping.
- Use Mahalanobis distance to find multivariate outliers a univariate check would miss.
- Distinguish MCAR, MAR and MNAR, and test which one a column's missingness looks like.
- Impute missing values with mean/median, KNN and MICE, and judge the trade-offs.
- Measure imputation quality by hiding known values and comparing the estimates.

## Prerequisites

- Notebook 04_01 (distributions, standard deviation, skew) and 04_02 (hypothesis testing,
  which we reuse to test missingness).
- Session 2/3 pandas (filtering, `isna`, `groupby`, selecting columns).

## Index

1. [Why this matters for risk analysis](#sec1)
2. [Outlier detection and handling](#sec2)
3. [Missing data mechanisms (and how to test for them)](#sec3)
4. [Imputation strategies and trade-offs](#sec4)
5. [Application: measuring imputation quality](#sec5)
6. [Exercises](#exercises)
7. [Additional Exercises](#additional)
8. [Challenge](#challenge)
9. [Key Takeaways](#takeaways)
10. [Further Reading](#reading)

<a id="setup"></a>
# Section 0: Setup

The teaching **examples** use two purpose-built datasets that we generate with a function so we control exactly what is in them: a *Heritage Brew Collective* customer dataset (for outliers) and a customer dataset with deliberately injected missingness (for the mechanisms). The **exercises** use real data from the repo-root `datasets/` folder: `train_titanic.csv` (genuine missing values and skewed `Fare`) and `superstore.csv`.

Note: scikit-learn's `IterativeImputer` (MICE) is experimental, so it needs the `enable_iterative_imputer` import before use.

**Documentation:** [scikit-learn](https://scikit-learn.org/stable/) - the core library for the imputation used throughout this notebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.spatial.distance import mahalanobis
from sklearn.experimental import enable_iterative_imputer  # noqa: needed for IterativeImputer
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer

sns.set_theme(style='whitegrid')
np.random.seed(42)  # reproducible results throughout

# Or read directly from the public S3 bucket (no local file needed):
# titanic = pd.read_csv('https://rockborne-bucket-01-cbs.s3.eu-west-2.amazonaws.com/Data_Sources_CBS_Risk/Session_4/train_titanic.csv')
# ...or read the paths from a config file (the local read below stays the default):
# from config import session_datasets_http
# titanic = pd.read_csv(session_datasets_http["train_titanic"])
# Or from S3 with Spark, then to pandas (needs a SparkSession, e.g. on Databricks):
# titanic = spark.read.csv("s3://rockborne-bucket-01-cbs/Data_Sources_CBS_Risk/Session_4/train_titanic.csv", header=True, inferSchema=True).toPandas()
titanic = pd.read_csv('../datasets/Session_4/train_titanic.csv')
# Or read directly from the public S3 bucket (no local file needed):
# store = pd.read_csv('https://rockborne-bucket-01-cbs.s3.eu-west-2.amazonaws.com/Data_Sources_CBS_Risk/Session_4/superstore.csv', encoding='latin1')
# Or from S3 with Spark, then to pandas (needs a SparkSession, e.g. on Databricks):
# store = spark.read.csv("s3://rockborne-bucket-01-cbs/Data_Sources_CBS_Risk/Session_4/superstore.csv", header=True, inferSchema=True).toPandas()
store = pd.read_csv('../datasets/Session_4/superstore.csv', encoding='latin1')
print('titanic rows:', len(titanic), '| superstore rows:', len(store))

<a id="sec1"></a>
# Section 1: Why this matters for risk analysis

Data quality is not housekeeping: in risk work, bad data quietly corrupts every model and report built on top of it.

| Topic | Why a risk team cares |
|---|---|
| **Outliers** | A single mis-keyed exposure or a genuine extreme loss can dominate a mean, a VaR estimate or a model coefficient. We must find them and decide, deliberately, whether to keep, cap or investigate them. |
| **Missing data** | Credit and customer data is full of gaps. Whether a value is missing at random or because of what it would have been (for example, income left blank by the highest earners) changes whether our analysis is biased. |
| **Imputation** | How we fill gaps feeds straight into scorecards and capital models. A careless fill can shrink variance, hide risk, or bias a default rate. |

Getting these wrong does not throw an error; it produces a confident, wrong number. That is why the statistical reasoning here matters.

<a id="sec2"></a>
# Section 2: Outlier detection and handling

**Definition:** an **outlier** is a data point that differs markedly from the rest. Outliers arise from measurement errors, data-entry errors (a misplaced decimal), or true rare-but-legitimate anomalies.

- **Measurement errors**: mistakes made during data collection  
- **Data-entry errors**: typographical errors when values are recorded  
- **True anomalies**: rare yet legitimate observations that genuinely deviate from the general pattern  

**Example:** in a coffee shop's records, a customer whose spend-per-visit is logged as 80 when everyone else is around 5.

**Analogy:** a 2.4 metre tall person in a room of adults: you spot them instantly because they sit far from everyone else.

**Explanation:** outliers distort means, standard deviations, correlations and model coefficients, so they especially hurt linear models. We never delete them blindly: we **detect**, then **decide** (keep, investigate, remove, or cap). We work through the full cycle on a realistic dataset.


## Main Methods to Detect Outliers

Various techniques exist for detecting outliers. Below are three statistical approaches: the first two are univariate (they examine one column at a time), while the third is multivariate (it examines several columns together):

1. **Z-score method**  
   This measures how many standard deviations a data point lies from the mean. A common rule of thumb is to flag any point with a Z-score greater than 3 or less than -3 as an outlier.

2. **Interquartile range (IQR) method**  
   This defines outliers as values falling below the first quartile (Q1) or above the third quartile (Q3) by more than 1.5 × IQR. It is especially useful for data that are not normally distributed.

3. **Mahalanobis distance method**  
   This measures how far a point lies from the centre of the data cloud while accounting for the correlation between variables. It flags points whose *combination* of values is unusual even when each value on its own looks perfectly normal, so it catches multivariate outliers that a univariate check on each column would miss.


In the example that follows, we will explor this concept by manually insert a few outliers, and then locate them using visualisations and the Z-score method.



### Key Strategies for Handling Outliers

1. **Removal**  
   Delete clearly erroneous values (e.g., an age of 200). Suitable when the outlier stems from data-entry or measurement errors. Remove sparingly to avoid excessive data loss.

2. **Imputation / Capping**  
   Retain the record but limit its influence, for example by replacing values above the 99th percentile with the 99th-percentile value (Winsorisation) or by substituting the median. Use when the point is genuine yet its magnitude distorts the model.

3. **Transformation**  
   Apply a mathematical function—logarithm or square root compress extreme values. Ideal for naturally skewed distributions (e.g., income) where outliers are inherent.

4. **Separate Modelling**  
   Treat outliers as a distinct class when they represent events of interest, such as fraud or rare faults. Build a dedicated detection model rather than removing or altering these observations.

There is no universal remedy; the optimal technique depends on the data characteristics and the business objective.


## The scenario: Heritage Brew Collective

*Heritage Brew Collective* is a neighbourhood coffee shop. Over the past year, management collected customer-level data (age, loyalty membership, how often each customer visits, and how much they spend per visit) to model customer loyalty and churn. We generate that dataset with a function so we know exactly what 'clean' looks like; note that spend rises with visit frequency, so the two are correlated (this matters later for Mahalanobis distance).

In [ ]:
def generate_customer_data(num_customers):
    """Synthetic customer-level data for Heritage Brew Collective.
    Spend per visit rises with visit frequency, so the two are positively correlated."""
    visits = np.random.normal(8, 2, num_customers).clip(1).round(1)
    spend = (3 + 0.4 * visits + np.random.normal(0, 0.8, num_customers)).clip(1).round(2)
    return pd.DataFrame({
        'customer_id': range(num_customers),
        'Age': np.random.randint(18, 65, num_customers),
        'Loyalty_Member': np.random.choice(['Yes', 'No'], num_customers, p=[0.7, 0.3]),
        'Avg_Visits_Per_Month': visits,
        'Spend_Per_Visit': spend,
    })

customers = generate_customer_data(1000)
customers.head()

### Step 1: Inject a few known outliers

To demonstrate detection, we deliberately insert outliers at known rows: two single-variable data-entry errors (an impossibly high spend, an impossibly high visit count), and one **multivariate** outlier whose values are each plausible on their own (12 visits, spend of 5.5) but whose *combination* breaks the usual pattern (frequent visitors normally spend more, around 7.8 at 12 visits). We will see which methods catch which.

In [ ]:
# data-entry error: absurd spend
# data-entry error: absurd visit count
# multivariate outlier: many visits...
# ...but low spend for that many visits

### Step 2: Visualise

A boxplot shows single-variable outliers (the points beyond the whiskers); a scatter plot of the two correlated features shows the multivariate outlier sitting off the diagonal trend.

### Step 3: Detect with the z-score and IQR methods

Both are univariate (they look at one column at a time) and return the row indices they flag. The **z-score** rule flags `|z| > 3`; the **IQR** rule flags values beyond `Q1 - 1.5 x IQR` or `Q3 + 1.5 x IQR` (good for skewed data, since quartiles are robust).

### Step 4: Add Mahalanobis distance for the multivariate outlier

**Definition:** **Mahalanobis distance** measures how far a point sits from the centre of the data *cloud* while accounting for the correlation between variables and the spread of each. Where a z-score treats every axis separately, Mahalanobis rescales by the covariance, so "far" is measured relative to the shape of the whole cloud.

**Example:** the customer at row 30 makes 12 visits a month but spends only 5.5 per visit. Twelve visits is unremarkable and a spend of 5.5 is unremarkable, but heavy visitors normally spend around 7.8, so the *pair* of values is odd. Mahalanobis flags row 30 while the z-score and IQR, looking at one column at a time, do not.

**Analogy:** a person who is 2 metres tall is unusual, and a person weighing 60 kg is unusual for that height, yet 2 m with 100 kg together is perfectly normal while 2 m with 60 kg is not. Judging height and weight *together*, allowing for the fact that they rise together, is exactly what Mahalanobis does; checking each on its own would miss the odd combination.

**Explanation:** row 30 is ordinary on each axis (so the z-score misses it) but unusual in combination, which Mahalanobis catches. A point is flagged when its distance exceeds the square root of the 97.5th percentile of a chi-square with degrees of freedom equal to the number of features. Because the classical covariance is itself distorted by the gross data-entry errors, we first remove those (rows 10 and 20) and then look for subtle multivariate outliers, which is the natural order of work anyway.

### Why row 30 is an outlier even though its values look normal

Spend rises with visits, so the data forms a tilted oval (few visits/low spend through to many visits/high spend). Row 30 has 12 visits and 5.5 spend: each value on its own is normal, which is why the z-score and IQR miss it. But heavy visitors normally spend around 7.8, not 5.5, so row 30 sits off the diagonal, inside each axis' range but outside the cloud. Mahalanobis measures distance along the shape of that oval, so it flags row 30 (distance about 3.5, cutoff about 2.72). The lesson: a multivariate outlier can hide inside perfectly normal single-column values.

In [ ]:
# Remove the gross single-variable errors first, then look for multivariate ones

### Choosing and interpreting alpha

`alpha` is not the cutoff; the cutoff distance is derived from it via `np.sqrt(stats.chi2.ppf(1 - alpha, df=n_features))`. Read `alpha` as the false-positive rate you accept: with `alpha = 0.025` you expect about 2.5% of normal rows to be flagged anyway. Larger alpha means a lower cutoff and more points flagged (more false alarms); smaller alpha means a higher cutoff and fewer flagged (you miss more). Typical values are 0.05, 0.025 or 0.01; the choice is a business trade-off between false alarms and misses.

### Step 5: Handle the outliers (remove or cap)

Once flagged and investigated, we either **remove** the rows or **cap** them at a sensible value (the median of the clean data, say). Capping keeps the row (and its other fields) while neutralising the extreme value. We write both as reusable functions.

### Step 6: Confirm the fix

Re-plotting the spend boxplot after removing the gross outliers shows a sensible spread again: the detect-then-fix loop worked.

### Try it yourself

Use `detect_outliers_iqr` on the titanic `Fare` column (pass `features=['Fare']`). How many rows does it flag?

In [ ]:
# Your turn. Write your solution here:

<a id="sec3"></a>
# Section 3: Missing data mechanisms (and how to test for them)

**Definition:** the *mechanism* of missingness is the process that decides which values are absent. It matters far more than how many are missing, because it determines whether ignoring or filling the gaps will bias your results.

**Example:** in a customer dataset, `Spend_Per_Visit` could be blank at random, or blank mainly for older customers, or blank mainly for the biggest spenders. Same number of blanks, very different consequences.

**Analogy:** missing jigsaw pieces. If pieces fell out at random you can still guess the picture; if every missing piece is from the same corner, that corner is unknowable.

**Explanation:** there are three mechanisms, defined by *what the missingness depends on*. Below we define each, then build a controlled dataset to show how to **test** which one we are likely facing.

### The three mechanisms

**MCAR - Missing Completely At Random.** The chance of being missing is unrelated to any data, observed or not (a dropped sample, a transfer error). The observed rows are still a fair sample, so dropping them is unbiased (just wasteful). *Analogy:* a few survey pages randomly slip out of the folder on the way to the office; which ones fell out has nothing to do with what was written on them.

**MAR - Missing At Random.** The chance of being missing depends only on **observed** variables, not on the missing value itself (older customers more often leave spend blank, and we know each customer's age). This is the practical sweet spot: because the missingness is explained by variables we can see, model-based imputation (KNN, MICE) can correct the bias. *Analogy:* younger customers skip the income box, but you know everyone's age, so the gap is explainable from what you can see.

**MNAR - Missing Not At Random.** The chance of being missing depends on the **missing value itself** (the biggest spenders hide their spend). This is the hardest case: no imputation from the observed data can fully fix it, because the reason for missingness is invisible. *Analogy:* the top earners refuse to state their income because it is high, so the cause of the gap is the hidden value itself.

In one line: MCAR is missing for no reason, MAR is missing for a reason you can see, and MNAR is missing for a reason you cannot.

### How do we test which mechanism it is?

The key idea: build a **missingness indicator** (1 if the value is missing, else 0) and test whether it is **associated with the observed variables**.

- If the indicator is **related to an observed variable** (for example, rows with missing spend have a significantly different mean age), the missingness depends on observed data, so it is **not MCAR**; it looks like **MAR**. We test this with a two-sample t-test (Session 04_02) comparing each observed variable between the missing and non-missing groups (use a chi-square test for categorical variables, or a logistic regression of the indicator for all of them at once).
- If the indicator is **unrelated to every observed variable**, the data is consistent with **MCAR** (the formal omnibus version is **Little's MCAR test**).
- **The catch:** MAR and MNAR look identical in the observed data. You cannot detect MNAR from the data alone, because the thing driving the missingness is the value you cannot see. Distinguishing them needs domain knowledge. We will demonstrate this directly.

We generate a clean customer dataset, then inject missingness into `Spend_Per_Visit` under each mechanism with a function, so we know the truth and can see what the test reveals.

In [ ]:
# older -> more likely missing (observed)
# higher spend -> more likely missing (hidden)

### The diagnostic test

The recipe is always the same: build a missingness indicator (1 if the target is missing, else 0) and test whether it is associated with the observed variables. Which test you use depends on the type of the observed variable:

- **Numeric observed variable -> two-sample t-test.** Split the rows into "target missing" and "target present" and compare the mean of the observed variable across the two groups. A small p-value means the groups differ, so the missingness is associated with that variable. This is the test our helper below uses.
- **Categorical observed variable -> chi-square test of independence.** Cross-tabulate the missingness flag against the category and test whether the two are independent (the same chi-square test from 04_02). A small p-value again means the missingness is associated with that variable.
- **All observed variables at once -> logistic regression.** Regress the 0/1 indicator on every observed variable together; any significant coefficient means the missingness depends on observed data. This is the joint version, handling numeric and categorical variables in one model.

In every case a significant result rules out MCAR. One caveat on the t-test: it only detects a difference in **means**, so it can miss a dependency that shows up in the variance or shape rather than the average.

The helper below applies the t-test version to each observed numeric variable for a target column; a small p-value flags an association that rules out MCAR.

### Reading the results

- **MCAR:** no observed variable is related to the missingness (all p-values large), exactly as expected.
- **MAR:** `Age` comes out significantly related (the rows with missing spend are older), which correctly tells us the missingness depends on an observed variable.
- **MNAR:** the observed variables look *unrelated*, so from the data alone MNAR is indistinguishable from MCAR. But because we injected it, we can peek at the true (hidden) spend values and confirm the missingness really depended on them.

Note the asymmetry: this test can rule MCAR out (a significant result), but it can never rule it in. A clean result only means "consistent with MCAR", and is equally consistent with MNAR; only domain knowledge can decide.

In [ ]:
# Peeking at the hidden truth for the MNAR scenario (only possible because we injected it)

### Try it yourself

Apply `diagnose_missingness` to the real titanic data: is the missingness in `Age` related to the observed `Fare` and `Pclass`? (Use `target='Age'`, `observed=['Fare', 'Pclass']`.)

In [ ]:
# Your turn. Write your solution here:

<a id="sec4"></a>
# Section 4: Imputation strategies and trade-offs

**Definition:** **imputation** fills missing values with estimates so the dataset is complete enough to analyse or model.

**Example:** replacing a missing spend with the median spend, or with a value predicted from the customer's age and visit frequency.

**Analogy:** an art restorer repainting a damaged patch. A flat grey fill is quick but obvious; matching the surrounding brushwork is more work but far more faithful.

**Explanation:** the methods below range from quick-and-crude to model-based. The key trade-off is **bias vs variance**: simple fills are easy but shrink variability and distort relationships; model-based fills are more faithful but assume the data is MAR.

## 4.1 Simple imputation (mean / median / mode)

Replace every gap with one summary value: the **mean** or **median** for numbers, the **mode** for categories, via `SimpleImputer`. Fast and transparent, but it **shrinks the variance** (many identical values), weakens correlations, and ignores the other columns. Median is safer than mean on skewed data. We show the variance shrinkage on the MAR scenario.

## 4.2 KNN imputation

**K-nearest-neighbours imputation** (`KNNImputer`) fills a gap with the average of the `k` most similar rows (nearest on the other columns). It uses the relationships between columns, so it is far more faithful than a single value, but it is slower, needs the features on a comparable scale, and assumes MAR.

**Choosing k.** The number of neighbours `k` trades noise against over-smoothing: a small k (1 to 3) follows local structure but is noisy and outlier-sensitive, while a large k is stable but drifts toward the global mean (at k = all rows, KNN imputation is just mean imputation). A default of 5 to 10 is usually fine (scikit-learn defaults to 5, we use 10 here). Rather than guess, choose k with the same hide-and-score RMSE from Section 5: try a few values and keep the lowest. Bear in mind k interacts with scaling, since neighbours are found by distance across the other columns.

## 4.3 MICE (Multiple Imputation by Chained Equations)

**MICE** (`IterativeImputer`) models each column with missing values as a function of the others and iterates until the estimates stabilise. Usually the most accurate for MAR data and it preserves relationships, but it is the most computationally intensive and also assumes MAR.

| Method | Strength | Weakness |
|---|---|---|
| Mean / median | fast, simple, transparent | shrinks variance, ignores other columns |
| KNN | uses inter-column structure | slower, scale-sensitive, assumes MAR |
| MICE (Iterative) | most faithful for MAR, keeps relationships | slowest, assumes MAR |
| Any of the above | (none can fix MNAR bias on its own) | |


<a id="sec5"></a>
# Section 5: Application - measuring imputation quality

How do we know which imputation method is best for a given dataset? We hide values we actually know, impute them, and compare the estimates to the truth with the root mean squared error (RMSE). Lower RMSE means a more faithful fill.

We use four complete, related numeric columns from the superstore data (Sales, Quantity, Discount, Profit) and build up to the comparison in three steps:

1. **Split into truth and a gappy copy.** Keep the real, complete Profit column as the truth, then copy it and hide 20% of its values at random (MCAR), so we know the correct answer for every gap we create.
2. **Define RMSE** to score each method against the truth on exactly those hidden cells.
3. **Compare the three methods** on the same holes: first the single RMSE score (in dollars), then the actual values each one filled in.

In [ ]:
# impute the 'Profit' column
# 20% MCAR holes
# One row per hidden Profit cell: the true value (in dollars) next to each method's fill

In [ ]:
# Choosing k empirically: reuse the Section 5 hide-and-score rmse() and try several k

### Reading the table

Each row is one customer whose Profit we hid, shown next to what each method guessed (in dollars).

- **Mean imputation** fills the identical value (about $27, the average profit) into every gap. It ignores the other columns entirely, so the whole column becomes constant and its RMSE is the worst.
- **KNN and MICE** give a different, tailored estimate for each row, because they use the other columns (Sales, Quantity, Discount) to inform the guess, so their values track the true Profit more closely and their RMSE is lower (see row 0: true 6.87, KNN 6.84, mean a useless 27.04).

This is what the RMSE summarised in a single number: the methods that use the other columns win. (KNN ideally wants the features on a comparable scale, as noted in 4.2; we keep raw dollars here for readability, which is enough to make the point.)

### Why MNAR cannot be fixed by imputation

When data is missing *because* of its own value, even good imputation is biased. We hide the **top 20%** of profits (an MNAR pattern, as if the biggest figures were withheld), then mean-impute and compare to the true mean.

In [ ]:
# top 20% are 'missing' (MNAR)
# mean fill uses only what we can see

<a id="exercises"></a>
# Section 6: Exercises

These exercises use the real `titanic` and `superstore` data, and the functions defined above.

### Exercise 1: Outliers with z-score

Using `detect_outliers_zscore`, how many outliers are there in the superstore `Profit` column (pass `features=['Profit']`)? Print the count.

In [ ]:
# Your turn. Write your solution here:

### Exercise 2: Diagnose missingness

Use `diagnose_missingness` on titanic with `target='Age'` and `observed=['SibSp', 'Parch']`. Is `Age`-missingness related to either? What mechanism does that suggest?

In [ ]:
# Your turn. Write your solution here:

### Exercise 3: Compare two imputations of Age

Fill the titanic `Age` column two ways: with the global median, and with the median within each `Pclass` group (`groupby('Pclass')...transform('median')`). Print the overall mean Age after each. Which uses more information?

In [ ]:
# Your turn. Write your solution here:

<a id="additional"></a>
## Additional Exercises

### Exercise A1: IQR vs z-score on skewed data

On the superstore `Sales` column (right-skewed), count the outliers flagged by IQR and by z-score (use the functions with `features=['Sales']`). Which flags more, and why is IQR more trustworthy here?

In [ ]:
# Your turn. Write your solution here:

### Exercise A2: Multivariate outliers with Mahalanobis

Use `detect_outliers_mahalanobis` on the superstore `['Sales', 'Profit', 'Quantity']` columns and print how many rows are flagged.

In [ ]:
# Your turn. Write your solution here:

<a id="challenge"></a>
## Challenge (optional): a data-quality audit of the titanic data

Produce a short data-quality audit. Work through the four parts; the coach answer shows one complete solution.

**Part 1: Outliers.** Use `detect_outliers_iqr` on `Fare`, report how many and inspect the largest few.

**Part 2: Missingness.** Report the share missing for `Age`, `Cabin` and `Embarked`.

**Part 3: Test the mechanism.** Use `diagnose_missingness` on `Age` against `Fare` and `Pclass`, and state whether `Age` looks MCAR, MAR or MNAR (and why you cannot fully rule out MNAR).

**Part 4: Impute and compare.** Fill `Age` with the global median and the per-`Pclass` median, and compare the variance of each to the original.

In [ ]:
# Your turn. Write your solution here:

<a id="takeaways"></a>
## Key Takeaways

| Concept / command | What it does |
|---|---|
| z-score (`|z|>3`) | Univariate outliers; assumes a symmetric distribution |
| IQR rule (`Q1-1.5*IQR`, `Q3+1.5*IQR`) | Robust univariate outliers; good for skewed data |
| Mahalanobis distance + chi-square cutoff | Multivariate outliers off the correlation structure |
| remove / cap | Two ways to handle flagged outliers once investigated |
| MCAR | Missing unrelated to anything; dropping rows is unbiased |
| MAR | Missing depends on observed variables; model-based imputation can fix it |
| MNAR | Missing depends on the hidden value; cannot be detected from observed data alone |
| missing-indicator t-test | Tests whether missingness is related to observed variables (MCAR vs MAR) |
| `SimpleImputer` / `KNNImputer` / `IterativeImputer` | Mean-median / KNN / MICE imputation |
| Hide-and-score (RMSE) | Measure imputation quality on values you actually know |


## Conclusion

You can now detect outliers in one and many dimensions and fix them, reason about and **test** why data is missing, impute gaps with methods of increasing sophistication, and measure how good an imputation really is. Crucially, you can recognise MNAR situations that no imputation can rescue. The next notebook looks at relationships between variables through correlation analysis.

<a id="reading"></a>
## Further Reading & Resources

- [scikit-learn: Imputation of missing values](https://scikit-learn.org/stable/modules/impute.html) SimpleImputer, KNNImputer, IterativeImputer.
- [scikit-learn IterativeImputer (MICE)](https://scikit-learn.org/stable/modules/generated/sklearn.impute.IterativeImputer.html) the chained-equations imputer.
- [Missing data mechanisms (MCAR, MAR, MNAR)](https://stefvanbuuren.name/fimd/) Flexible Imputation of Missing Data, the standard reference (also covers Little's MCAR test).